This notebook showcases TxConformal for controlling other error metrics.

In [ ]:
import os
import numpy as np 
import pickle
import pandas as pd  
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

from txconformal import TxConformal
from txconformal.features.providers import FeaturesProvider

In [ ]:
task_type = 'classification'  
drug_encoding = 'Morgan'
score_type = 'bnn_loss'
path = '/Users/yjinstat/Desktop/Research/collaborations/cf_drug_discovery/data/admet/' + task_type + '/' + score_type + '/' + drug_encoding + '/'

file_list = os.listdir(path)
file_list = [x for x in file_list if 'scaffold' in x]

head_list = list(set([x.partition('_scaffold')[0] for x in file_list]))
 
file_names = [x for x in file_list if x.partition('_scaffold')[0] == 'cyp2d6_veith']
ex_file_name = [x for x in file_names if 'e(x)' in x][0]
dat_file_name = [x for x in file_names if 'e(x)' not in x and 'hid' not in x][0] 
hid_file_name = [x for x in file_names if 'hid' in x][0] 

res = pd.read_csv(os.path.join(path, dat_file_name))
exs = pd.read_csv(os.path.join(path, ex_file_name))
with open(os.path.join(path, hid_file_name), 'rb') as file:
    hid_feature = pickle.load(file)

res_cal_all = res[res['split'] == 'calib']
res_test_all = res[res['split'] == 'test']
emb_cal_all = hid_feature[res['split'] == 'calib', :]
emb_test_all = hid_feature[res['split'] == 'test', :] 


### Task 1: FDR control

Address domain question such as: How many candidates can I select until I make 10% mistakes? (FDR target = 0.1)

In [3]:
seed = 3
res_cal, _, emb_cal, _ = train_test_split(res_cal_all, emb_cal_all, test_size=0.2, random_state=seed)
res_test, _, emb_test, _ = train_test_split(res_test_all, emb_test_all, test_size=0.2, random_state=seed)
y_calib = res_cal['Label'].values
f_calib = res_cal['pred'].values
f_test = res_test['pred'].values

# feature preparation
prov = FeaturesProvider(f_calib=f_calib, f_test=f_test)
prov.prepare() 

for FDR_target in np.linspace(0.1, 0.9, 9):
    # run TxConformal to fit weights
    txc = TxConformal(score_name="clip")
    txc.fit(prov, y_calib, cutoff=0.5,print_level=-1)
    # apply selection with BH for FDR control at the target level
    sel_res = txc.select(method="bh", alpha=FDR_target)

    sel_idx = sel_res.idx
    fdp_est = np.sum(res_test.iloc[sel_idx]['Label'] == 0) / len(sel_idx)  if len(sel_idx) > 0 else 0.0
    print(f"FDR target: {FDR_target:.2f} | selected: {len(sel_idx)} | estimated FDP: {fdp_est:.3f}")

FDR target: 0.10 | selected: 26 | estimated FDP: 0.115
FDR target: 0.20 | selected: 61 | estimated FDP: 0.246
FDR target: 0.30 | selected: 99 | estimated FDP: 0.354
FDR target: 0.40 | selected: 161 | estimated FDP: 0.422
FDR target: 0.50 | selected: 217 | estimated FDP: 0.512
FDR target: 0.60 | selected: 313 | estimated FDP: 0.613
FDR target: 0.70 | selected: 506 | estimated FDP: 0.690
FDR target: 0.80 | selected: 1031 | estimated FDP: 0.819
FDR target: 0.90 | selected: 1050 | estimated FDP: 0.822


Validate the FDR control of TxConformal (10 seeds for convenience)

In [6]:
FDR_target = 0.1
all_fdp = []

for seed in range(10):
    res_cal, _, emb_cal, _ = train_test_split(res_cal_all, emb_cal_all, test_size=0.2, random_state=seed)
    res_test, _, emb_test, _ = train_test_split(res_test_all, emb_test_all, test_size=0.2, random_state=seed)
    y_calib = res_cal['Label'].values
    f_calib = res_cal['pred'].values
    f_test = res_test['pred'].values

    # use default setup for feature preparation
    prov = FeaturesProvider(f_calib=f_calib, f_test=f_test, E_calib=emb_cal, E_test=emb_test)
    prov.prepare()
    # run TxConformal to fit weights
    txc = TxConformal(score_name="clip")
    txc.fit(prov, y_calib, cutoff=0.5, print_level=-1)
    # apply selection with BH for FDR control at the target level
    sel_res = txc.select(method="bh", alpha=FDR_target)

    # evaluate the FDP
    sel_idx = sel_res.idx
    fdp_est = np.sum(res_test.iloc[sel_idx]['Label'] == 0) / len(sel_idx)  if len(sel_idx) > 0 else 0.0
    print(f"seed: {seed} | selected: {len(sel_idx)} | estimated FDP: {fdp_est:.3f}")
    all_fdp.append(fdp_est)

print(f"Target FDR: {FDR_target:.3f} | Average FDP: {np.mean(all_fdp):.3f}")

seed: 0 | selected: 29 | estimated FDP: 0.103
seed: 1 | selected: 30 | estimated FDP: 0.100
seed: 2 | selected: 28 | estimated FDP: 0.071
seed: 3 | selected: 26 | estimated FDP: 0.115
seed: 4 | selected: 37 | estimated FDP: 0.081
seed: 5 | selected: 23 | estimated FDP: 0.174
seed: 6 | selected: 29 | estimated FDP: 0.069
seed: 7 | selected: 38 | estimated FDP: 0.211
seed: 8 | selected: 20 | estimated FDP: 0.150
seed: 9 | selected: 30 | estimated FDP: 0.100
Target FDR: 0.100 | Average FDP: 0.117


### Task 2: Ensure number of true positives

Addressing domain question like: How many candidates should I select to ensure at least 35 positive ones?

In [7]:
prov = FeaturesProvider(f_calib=f_calib, f_test=f_test, E_calib=emb_cal, E_test=emb_test)
prov.prepare() 
txc = TxConformal(score_name="clip", random_state=2)
txc.fit(prov, y_calib, cutoff=0.5, print_level=-1)

for kk in np.linspace(5, 50, num=10):

    sel_res = txc.select(method = 'tp_min', K = kk)
    sel_idx = sel_res.idx
    tp_count = np.sum(res_test.iloc[sel_idx]['Label']==1) if len(sel_idx)>0 else 0.0
    print(f"Target TP: {int(kk)} | Realized TP: {tp_count} | Number of selection: {len(sel_idx)}")
 

Target TP: 5 | Realized TP: 6 | Number of selection: 6
Target TP: 10 | Realized TP: 10 | Number of selection: 11
Target TP: 15 | Realized TP: 15 | Number of selection: 16
Target TP: 20 | Realized TP: 20 | Number of selection: 22
Target TP: 25 | Realized TP: 25 | Number of selection: 28
Target TP: 30 | Realized TP: 30 | Number of selection: 35
Target TP: 35 | Realized TP: 35 | Number of selection: 42
Target TP: 40 | Realized TP: 40 | Number of selection: 50
Target TP: 45 | Realized TP: 43 | Number of selection: 57
Target TP: 50 | Realized TP: 48 | Number of selection: 66


### Task 3: control number of false positives

Addressing domain questions like: how many candidates can I test until I make 20 mistakes?

In [8]:
for kk in np.linspace(5, 50, num=10):

    sel_res = txc.select(method = 'fp_budget', K = kk)
    sel_idx = sel_res.idx
    tp_count = np.sum(res_test.iloc[sel_idx]['Label']==0) if len(sel_idx)>0 else 0.0
    print(f"Target FP: {int(kk)} | Realized FP: {tp_count} | Number of selection: {len(sel_idx)}")

Target FP: 5 | Realized FP: 5 | Number of selection: 35
Target FP: 10 | Realized FP: 11 | Number of selection: 52
Target FP: 15 | Realized FP: 18 | Number of selection: 64
Target FP: 20 | Realized FP: 24 | Number of selection: 77
Target FP: 25 | Realized FP: 26 | Number of selection: 82
Target FP: 30 | Realized FP: 32 | Number of selection: 91
Target FP: 35 | Realized FP: 38 | Number of selection: 100
Target FP: 40 | Realized FP: 47 | Number of selection: 119
Target FP: 45 | Realized FP: 48 | Number of selection: 121
Target FP: 50 | Realized FP: 51 | Number of selection: 126


### Task 4: Estimate number of false positives in top-K selection

Addressing domain questions like: I want to screen my top 100 candidates. How many of them are expected to be false leads?

In [19]:
for kk in np.linspace(100, 400, num = 7):
    sel_res = txc.select(method = 'top_K', K = kk) 
    sel_idx = sel_res.idx
    true_fdp = np.sum(res_test.iloc[sel_idx]['Label']==0) / len(sel_idx) if len(sel_idx)>0 else 0.0
    print(f"K: {int(kk)} | Estimated FDP: {sel_res.fdp_est:.3f} | True FDP: {true_fdp:.3f}")

K: 100 | Estimated FDP: 0.333 | True FDP: 0.380
K: 150 | Estimated FDP: 0.436 | True FDP: 0.440
K: 200 | Estimated FDP: 0.496 | True FDP: 0.525
K: 250 | Estimated FDP: 0.578 | True FDP: 0.588
K: 300 | Estimated FDP: 0.632 | True FDP: 0.637
K: 350 | Estimated FDP: 0.661 | True FDP: 0.669
K: 400 | Estimated FDP: 0.666 | True FDP: 0.682
